# example / wasm / accumulate

Add a fitted line to each input and keep a running total, as a WebAssembly module.

| Term | In This Notebook |
|---|---|
| **Intent** | Show a stateful `wasm` model end to end: fit, build the module, accept, replay the golden and register |
| **Project Flavour** | `wasm` (`model.md` 3.2) |
| **Data Splits** | `training` ten labelled points on a line with fixed noise, to fit; `validation` five more; `backtest` ten further out, held back for acceptance; `golden` the exact samples that become `golden.json` |

## Setup

- **Stage:** none, preparation for the stages below.
- **Intent:** put `workbench` on the path, load `env`, and declare the model name, version and dataset names as constants (`model.md` 6.6).

In [ ]:
import json
from os.path import join

import numpy as np

from workbench.config import NOTEBOOK_DIR
from workbench.data import ensure_data, golden_data
from workbench.env import init_env
from workbench.project import provenance, record_project, scratch_project_dir

env = init_env()

WORKSPACE = "example/wasm/accumulate"
MODEL_NAME = "example_wasm_accumulate"
TARGETS = [{"kind": "served"}, {"kind": "embedded", "language": "python"}, {"kind": "embedded", "language": "go"}, {"kind": "embedded", "language": "rust"}]
TRAINING = "wasm_accumulate_training_2026-11-03_2026-11-03"
VALIDATION = "wasm_accumulate_validation_2026-11-03_2026-11-03"
BACKTEST = "wasm_accumulate_backtest_2026-11-03_2026-11-03"
GOLDEN = "wasm_accumulate_golden"
INPUT = "example_wasm_in"
OUTPUT = "example_wasm_out"

## Capture Data

- **Stage:** 1, capture data (`model.md` 7.1).
- **Intent:** write the `training`, `validation` and `backtest` files only when they are absent (`model.md` 6.5). The source is synthetic, so no query runs.

In [ ]:
NOISE = [0.04, -0.03, 0.02, -0.05, 0.01, 0.03, -0.02, 0.05, -0.04, 0.0]


def labelled(first, count):
    xs = [float(x) for x in range(first, first + count)]
    return {INPUT: xs, OUTPUT: [round(2.5 * x + 1.0 + NOISE[int(x) % 10], 4) for x in xs]}


training_pl, training_path = ensure_data(WORKSPACE, "training", TRAINING, lambda: labelled(0, 10))
validation_pl, validation_path = ensure_data(WORKSPACE, "validation", VALIDATION, lambda: labelled(10, 5))
backtest_pl, backtest_path = ensure_data(WORKSPACE, "backtest", BACKTEST, lambda: labelled(15, 10))
print("training", training_pl.height, "validation", validation_pl.height, "backtest", backtest_pl.height)

## Fit

- **Stage:** 3, fit.
- **Intent:** fit a straight line on `training` only (R6). Its gain and offset are rounded to one place, which the module builds in.

In [ ]:
gain, offset = np.polyfit(training_pl[INPUT].to_numpy(), training_pl[OUTPUT].to_numpy(), 1)
GAIN, OFFSET = round(float(gain), 1), round(float(offset), 1)
print("fitted gain", GAIN, "offset", OFFSET)
assert (GAIN, OFFSET) == (2.5, 1.0), "the fit did not recover the line that made the data"

## Evaluate

- **Stage:** 4, evaluate.
- **Intent:** trace `accumulate`, the fitted line and the running total written once as a Python function, into the module (`model.md` 4.4), build the signature, run the module through the library host and check the `validation` and `backtest` error against the limit the notebook owns (R4, R14).

In [ ]:
from asystem.model import build_signature, evaluate_acceptance, measure, signature_input, signature_output, signature_state
from asystem.replay import model_predict
from workbench.wasm import build_wasm, trace


def accumulate(x, total, gain, offset):
    out = gain * x + offset
    return out, total + out


model_path = join(NOTEBOOK_DIR, WORKSPACE, "model.wasm")
with open(model_path, "wb") as model_file:
    model_file.write(build_wasm(2, trace(lambda x, total: accumulate(x, total, GAIN, OFFSET), 2)))

signature = build_signature(
    MODEL_NAME,
    [
        signature_input(INPUT, "none", "A number to scale."),
        signature_input("example_wasm_state_in", "none", "The running total from the previous call, absent on the first.", required=False, shape=[-1]),
    ],
    [
        signature_output(OUTPUT, "none", "The input scaled by the fitted gain plus the fitted offset.", datatype="FP64"),
        signature_output("example_wasm_state_out", "none", "The running total including this call.", datatype="FP64", required=False, shape=[-1]),
    ],
    state=[signature_state("total", "none", "The sum of every output since the first call.")],
)
predict = model_predict(model_path, "wasm", signature)

In [ ]:
def acceptance_checks():
    checks = []
    for split, frame in (("validation", validation_pl), ("backtest", backtest_pl)):
        predicted = [predict({INPUT: [x]})[OUTPUT][0] for x in frame[INPUT].to_list()]
        checks.append({"split": split, "output": OUTPUT, "name": "rmse", "value": measure(frame[OUTPUT].to_list(), predicted)["rmse"], "limit": 0.5, "direction": "max"})
    return checks


evaluation = evaluate_acceptance(acceptance_checks, {"validation": [validation_path], "backtest": [backtest_path]})
for entry in evaluation["checks"]:
    print(entry["split"], entry["output"], entry["name"], "measured", entry["value"], "limit", entry["limit"], "passed", entry["passed"])

## Package

- **Stage:** 5, package.
- **Intent:** write the golden file, copy the module and the evaluation into a project, verify it and replay the golden through the library host (R1, R3, R14).

In [ ]:
def golden_samples():
    xs = [1.0, 2.0, 3.0]
    totals, outs, states = [0.0], [], [None]
    for x in xs:
        out, total = accumulate(x, totals[-1], GAIN, OFFSET)
        outs.append(out)
        totals.append(total)
        states.append(str(totals[-1]))
    return {
        INPUT: [*xs, None],
        "example_wasm_state_in": [*states[:3], None],
        OUTPUT: [*outs, None],
        "example_wasm_state_out": [*[str(total) for total in totals[1:]], None],
        "expect": ["ok", "ok", "ok", "error"],
    }

In [ ]:
from asystem.model import package_project, register_preview
from asystem.replay import require_replay

golden_pl, golden_path = golden_data(WORKSPACE, GOLDEN, golden_samples)

project_dir = scratch_project_dir(WORKSPACE)
manifest = package_project(
    project_dir,
    MODEL_NAME,
    "wasm",
    signature,
    golden_pl.iter_rows(named=True),
    provenance(
        "manual",
        "synthetic line, training split",
        [training_path, golden_path],
        "least squares line, gain and offset built into the module",
        implementation="WebAssembly traced from a Python function by workbench.wasm.trace and built by build_wasm",
    ),
    model_path=model_path,
    evaluation=evaluation,
    sequence="a running total over three calls",
)
print(json.dumps(manifest, indent=2))

require_replay(project_dir)
metadata = record_project(WORKSPACE, project_dir, TARGETS)
print("replayed the golden sequences exactly through the shipped runtime")

## Register

- **Stage:** 6, register.
- **Intent:** register the verified project with the alias `preview`. A person sets `release` after review and the parity gate (R5).

In [ ]:
version = register_preview(project_dir, MODEL_NAME, env["MLFLOW_TRACKING_URI"], TARGETS, metadata["snapshot_version"])
print("registered version", version)